# 🧠 Laboratorio 2 – Ranked Retrieval with TF-IDF and Cosine Similarity

**Corso:** Information Retrieval – Laurea Magistrale in Informatica  
**Università:** Roma “Tor Vergata”  
**Docente:** Danilo Croce

---

## 🎯 Obiettivo del laboratorio

In questo laboratorio introduciamo il **ranked retrieval**, cioè un modello di retrieval in cui i documenti non vengono più considerati semplicemente come *matching* o *non matching* rispetto a una query, ma vengono **ordinati per rilevanza stimata**.

Nei laboratori precedenti abbiamo costruito sistemi di retrieval booleano:
- un documento soddisfa oppure non soddisfa una query
- il risultato è un insieme di documenti
- non c'è un vero ordinamento per rilevanza

In questo notebook facciamo un passo ulteriore:
- rappresentiamo documenti e query come **vettori**
- pesiamo i termini con **TF-IDF**
- confrontiamo query e documenti usando la **cosine similarity**
- otteniamo un **ranking** dei documenti

---

## 📌 Idea chiave

Nel retrieval booleano il sistema decide:

- **match**
- **no match**

Nel retrieval vettoriale il sistema assegna invece a ogni documento un **punteggio numerico** rispetto alla query.

Più alto è il punteggio, più il documento viene considerato rilevante.

Questa idea è importante perché risponde a un limite classico del modello booleano:

- alcune query restituiscono troppi documenti
- altre troppo pochi
- spesso manca un criterio per mostrare prima i risultati migliori

Con il ranked retrieval, invece, possiamo sempre mostrare per primi i documenti con score più alto.

---

## 🧩 Concetti principali del laboratorio

Nel notebook introdurremo progressivamente:

- **bag of words**
- **term frequency (tf)**
- **document frequency (df)**
- **inverse document frequency (idf)**
- **peso tf-idf**
- **rappresentazione vettoriale di documenti e query**
- **cosine similarity**
- **ranking dei documenti**

---

## 🛠️ Struttura del notebook

Procederemo in questo ordine:

1. caricamento di una collezione reale  
2. preprocessing dei documenti  
3. costruzione del vocabolario  
4. calcolo di tf, df e idf  
5. costruzione dei vettori dei documenti  
6. costruzione del vettore di query  
7. ranking con cosine similarity  
8. piccola estensione: combinare **body** e **title** con pesi diversi

---

## 🎓 Risultati di apprendimento attesi

Al termine del laboratorio lo studente dovrebbe essere in grado di:

- spiegare la differenza tra retrieval booleano e retrieval con ranking
- rappresentare documenti e query come vettori di pesi
- calcolare pesi **tf-idf**
- usare la **cosine similarity** per confrontare documenti e query
- comprendere il ruolo della normalizzazione dei vettori
- interpretare qualitativamente il ranking restituito dal sistema

In [ ]:
import math
import re
from collections import Counter, defaultdict

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

from sklearn.datasets import fetch_20newsgroups

In [ ]:
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## Perché passare dal booleano al ranking?

Nel modello booleano una query come:

`graphics AND file`

restituisce i documenti che soddisfano esattamente la condizione.

Questo modello è molto utile per capire:
- l'indicizzazione
- le postings lists
- il query processing

Tuttavia presenta alcuni limiti pratici:

- non distingue tra documenti “molto rilevanti” e documenti “appena compatibili”
- non ordina i risultati
- può produrre il classico problema del **feast or famine**:
  - troppi risultati
  - oppure zero risultati

Nel ranked retrieval affrontiamo il problema in modo diverso:

- non chiediamo più solo se il documento matcha la query
- chiediamo **quanto** il documento è vicino alla query

Per fare questo, dobbiamo costruire una rappresentazione numerica di documenti e query.

## Caricamento della collezione

Per mantenere continuità con il laboratorio precedente, useremo ancora un sottoinsieme del dataset **20 Newsgroups**.

Lavorare su una collezione reale è utile perché permette di osservare:

- rumore nei documenti
- effetti del preprocessing
- presenza di termini frequenti e rari
- comportamento reale del ranking

In [ ]:
def load_collection(categories=None, subset="train"):
    """
    Carica un sottoinsieme del dataset 20 Newsgroups.

    Parameters
    ----------
    categories : list[str] or None
        Categorie da caricare.
    subset : str
        Porzione del dataset: 'train' oppure 'test'.

    Returns
    -------
    tuple
        (documents, target_names)
    """
    if categories is None:
        categories = ["comp.graphics"]

    dataset = fetch_20newsgroups(
        subset=subset,
        categories=categories,
        remove=()
    )

    return dataset.data, dataset.target_names

In [ ]:
categories = ["comp.graphics"]

documents, target_names = load_collection(categories=categories, subset="train")

print("Selected categories:", categories)
print("Number of documents:", len(documents))

print("\nFirst raw document:\n")
print(documents[0][:1500])

Selected categories: ['comp.graphics']
Number of documents: 584

First raw document:

From: bbs.mirage@tsoft.net (Jerry Lee)
Subject: Cobra 2.0 1-b-1 Video card HELP ME!!!!
Organization: The TSoft BBS and Public Access Unix, +1 415 969 8238
Lines: 22

Does ANYONE out there in Net-land have any information on the Cobra 2.20 
card?  The sticker on the end of the card reads
        Model: Cobra 1-B-1
        Bios:  Cobra v2.20

I Havn't been able to find anything about it from anyone!  If you have 
any information on how to get a hold of the company which produces the 
card or know where any drivers are for it, PLEASE let me know!

As far as I can tell, it's a CGA card that is taking up 2 of my 16-bit 
ISA slots but when I enable the test patterns, it displays much more than 
the usualy 4 CGA colors... At least 16 from what I can count.. Thanks!

              .------------------------------------------.
              : Internet: jele@eis.calstate.edu          :
              :           

## Preprocessing

Anche nel ranked retrieval, la qualità della rappresentazione dipende fortemente dal preprocessing.

Useremo una pipeline semplice e coerente con i laboratori precedenti:

- rimozione dell’header
- lowercase
- normalizzazione dei numeri
- rimozione della punteggiatura
- tokenizzazione
- rimozione delle stopwords
- rimozione dei token troppo corti
- stemming

L’obiettivo è mantenere il notebook leggibile e mostrare chiaramente come la rappresentazione finale dei documenti dipenda dalle scelte di preprocessing.

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))


def remove_header(text):
    parts = text.split("\n\n", 1)
    if len(parts) == 2:
        return parts[1]
    return text


def convert_lower_case(text):
    return text.lower()


def convert_numbers(text):
    number_map = {
        "0": " zero ",
        "1": " one ",
        "2": " two ",
        "3": " three ",
        "4": " four ",
        "5": " five ",
        "6": " six ",
        "7": " seven ",
        "8": " eight ",
        "9": " nine ",
    }
    for digit, word in number_map.items():
        text = text.replace(digit, word)
    return text


def remove_punctuation(text):
    return re.sub(r"[^\w\s]", " ", text)


def remove_extra_spaces(text):
    return re.sub(r"\s+", " ", text).strip()


def tokenize(text):
    return word_tokenize(text)


def remove_stop_words(tokens):
    return [token for token in tokens if token not in stop_words]


def remove_single_characters(tokens):
    return [token for token in tokens if len(token) > 1]


def apply_stemming(tokens):
    return [stemmer.stem(token) for token in tokens]


def preprocess(text, is_query=False):
    if not is_query:
        text = remove_header(text)

    text = convert_lower_case(text)
    text = convert_numbers(text)
    text = remove_punctuation(text)
    text = remove_extra_spaces(text)

    tokens = tokenize(text)
    tokens = remove_stop_words(tokens)
    tokens = remove_single_characters(tokens)
    tokens = apply_stemming(tokens)

    return tokens

In [ ]:
raw_example = documents[0]
processed_example = preprocess(raw_example, is_query=False)

print("First 1200 raw characters:\n")
print(raw_example[:1200])

print("\n" + "=" * 80 + "\n")

print("First 80 preprocessed tokens:\n")
print(processed_example[:80])

First 1200 raw characters:

From: bbs.mirage@tsoft.net (Jerry Lee)
Subject: Cobra 2.0 1-b-1 Video card HELP ME!!!!
Organization: The TSoft BBS and Public Access Unix, +1 415 969 8238
Lines: 22

Does ANYONE out there in Net-land have any information on the Cobra 2.20 
card?  The sticker on the end of the card reads
        Model: Cobra 1-B-1
        Bios:  Cobra v2.20

I Havn't been able to find anything about it from anyone!  If you have 
any information on how to get a hold of the company which produces the 
card or know where any drivers are for it, PLEASE let me know!

As far as I can tell, it's a CGA card that is taking up 2 of my 16-bit 
ISA slots but when I enable the test patterns, it displays much more than 
the usualy 4 CGA colors... At least 16 from what I can count.. Thanks!

              .------------------------------------------.
              : Internet: jele@eis.calstate.edu          :
              :           bbs.mirage@gilligan.tsoft.net  :
              :          

## Dalla collezione ai documenti tokenizzati

Ora applichiamo il preprocessing a tutti i documenti.

Memorizzeremo i risultati in un dizionario che associa a ogni `docID` la lista dei token preprocessati.

Questa rappresentazione sarà la base per:
- costruire il vocabolario
- contare le frequenze dei termini
- costruire i vettori dei documenti

In [ ]:
def build_tokenized_documents(documents):
    """
    Applica il preprocessing a tutti i documenti della collezione.

    Returns
    -------
    dict[int, list[str]]
        docID -> lista di token preprocessati
    """
    tokenized_documents = {}

    for doc_id, text in enumerate(documents):
        tokenized_documents[doc_id] = preprocess(text, is_query=False)

    return tokenized_documents

In [ ]:
tokenized_documents = build_tokenized_documents(documents)

print("Number of preprocessed documents:", len(tokenized_documents))

for doc_id in range(3):
    print(f"\nDoc {doc_id} -> first 40 tokens:")
    print(tokenized_documents[doc_id][:40])

Number of preprocessed documents: 584

Doc 0 -> first 40 tokens:
['anyon', 'net', 'land', 'inform', 'cobra', 'two', 'two', 'zero', 'card', 'sticker', 'end', 'card', 'read', 'model', 'cobra', 'one', 'one', 'bio', 'cobra', 'two', 'two', 'zero', 'havn', 'abl', 'find', 'anyth', 'anyon', 'inform', 'get', 'hold', 'compani', 'produc', 'card', 'know', 'driver', 'pleas', 'let', 'know', 'far', 'tell']

Doc 1 -> first 40 tokens:
['hi', 'everyon', 'one', 'touch', 'problem', 'post', 'last', 'week', 'guess', 'question', 'clear', 'like', 'describ', 'detail', 'offset', 'ellips', 'locu', 'center', 'circl', 'roll', 'ellips', 'word', 'distanc', 'ellips', 'offset', 'everywher', 'problem', 'come', 'geometr', 'measur', 'probe', 'use', 'tip', 'probe', 'ball', 'comput', 'output', 'posit', 'ball', 'center']

Doc 2 -> first 40 tokens:
['hi', 'short', 'look', 'fast', 'assembl', 'code', 'line', 'circl', 'draw', 'svga', 'graphic', 'complet', 'think', 'simpl', 'fast', 'molecular', 'graphic', 'program', 'write', 'pc

## Bag of words model

Per il Vector Space Model useremo una rappresentazione **bag of words**.

Questo significa che:
- consideriamo quali termini compaiono nel documento
- consideriamo quante volte compaiono
- **non** consideriamo l’ordine con cui compaiono

Questa è una semplificazione importante.

Per esempio, le frasi:

- `john is quicker than mary`
- `mary is quicker than john`

risultano molto simili in una rappresentazione bag of words.

Perdiamo quindi informazione sull’ordine, ma otteniamo una rappresentazione più semplice e adatta al ranking vettoriale.

## Costruzione del vocabolario

Il primo passo verso la rappresentazione vettoriale consiste nel costruire il **vocabolario** della collezione, cioè l’insieme ordinato di tutti i termini distinti osservati nei documenti preprocessati.

Ogni termine del vocabolario corrisponderà a una dimensione dello spazio vettoriale.

In [ ]:
def build_vocabulary(tokenized_documents):
    """
    Costruisce il vocabolario globale della collezione.

    Returns
    -------
    list[str]
        Lista ordinata dei termini distinti
    """
    vocabulary = list(
        set(
            token
            for tokens in tokenized_documents.values()
            for token in tokens
        )
    )
    return vocabulary

In [ ]:
vocabulary = build_vocabulary(tokenized_documents)

print("Vocabulary size:", len(vocabulary))
print("\nFirst 40 vocabulary terms:")
print(vocabulary[:40])

Vocabulary size: 8603

First 40 vocabulary terms:
['techniqu', 'multipli', 'oasi', 'root', 'page', 'overlook', 'cologn', 'isu', 'automov', 'que', 'probabl', 'softlab', 'portland', 'weak', 'waldensoftwar', 'topolog', 'maxen', 'acorn', 'eas', 'cirru', 'copper', 'chert', 'ditolla', 'hotel', 'undelet', 'roy', 'saw', 'seek', 'hawnew', 'differend', 'curiou', 'john', 'msg', 'bind', 'alias', 'supercomput', 'nutshel', 'inbuilt', 'comparis', 'acc']


## Document frequency

Nel ranked retrieval è importante distinguere tra:

- **term frequency (tf)**: quante volte un termine compare in un documento
- **document frequency (df)**: in quanti documenti compare il termine

La document frequency è importante perché ci aiuta a distinguere:

- termini molto diffusi nella collezione
- termini più rari e quindi spesso più informativi

In [ ]:
def build_document_frequency(tokenized_documents):
    """
    Calcola la document frequency di ciascun termine.

    Returns
    -------
    dict[str, int]
        termine -> numero di documenti in cui compare
    """
    df = defaultdict(int)

    for tokens in tokenized_documents.values():
        seen_terms = set(tokens)
        for term in seen_terms:
            df[term] += 1

    return dict(df)

In [ ]:
document_frequency = build_document_frequency(tokenized_documents)

print("Vocabulary size from df:", len(document_frequency))

example_terms = ["graphic", "imag", "file", "format", "window"]

print("\nSome document frequencies:\n")
for term in example_terms:
    print(f"{term:12s} -> df = {document_frequency.get(term, 0)}")

Vocabulary size from df: 8603

Some document frequencies:

graphic      -> df = 190
imag         -> df = 139
file         -> df = 158
format       -> df = 71
window       -> df = 74


## Term frequency

La **term frequency** misura quante volte un termine compare in uno specifico documento.

Per esempio:
- se `graphic` compare 1 volta nel documento `d`, allora `tf(graphic, d) = 1`
- se compare 7 volte, allora `tf(graphic, d) = 7`

Nel ranked retrieval la frequenza conta, perché:
- un termine della query che compare molte volte in un documento tende a essere più rappresentativo
- ma la crescita della rilevanza non è perfettamente lineare

Per questo, invece del tf grezzo, spesso si usa una sua trasformazione logaritmica.

In [ ]:
def compute_term_frequencies(tokens):
    """
    Calcola la term frequency grezza di un documento.

    Returns
    -------
    collections.Counter
        termine -> numero di occorrenze nel documento
    """
    return Counter(tokens)

In [ ]:
doc_id = 0
tf_example = compute_term_frequencies(tokenized_documents[doc_id])

print(f"Document {doc_id} length:", len(tokenized_documents[doc_id]))
print("\nSome term frequencies in document 0:\n")

for term in ["cobra", "card", "inform", "driver", "graphic"]:
    print(f"{term:12s} -> tf = {tf_example.get(term, 0)}")

Document 0 length: 92

Some term frequencies in document 0:

cobra        -> tf = 3
card         -> tf = 4
inform       -> tf = 2
driver       -> tf = 1
graphic      -> tf = 0


## Peso logaritmico della term frequency

Una scelta classica nel Vector Space Model è usare il **log-frequency weighting**:

$$
w_{t,d}^{tf} =
\begin{cases}
1 + \log_{10}(tf_{t,d}) & \text{se } tf_{t,d} > 0 \\
0 & \text{altrimenti}
\end{cases}
$$

Questa trasformazione è utile perché:
- distingue tra presenza singola e presenza multipla del termine
- ma evita che frequenze molto alte dominino troppo il punteggio

In altre parole:
- passare da 1 a 2 occorrenze conta
- passare da 100 a 101 conta molto meno

In [ ]:
def log_tf_weight(tf):
    """
    Calcola il peso logaritmico della term frequency.

    Parameters
    ----------
    tf : int

    Returns
    -------
    float
    """
    if tf > 0:
        return 1.0 + math.log10(tf)
    return 0.0

In [ ]:
for value in [0, 1, 2, 5, 10, 100]:
    print(f"tf = {value:3d} -> log-tf weight = {log_tf_weight(value):.4f}")

tf =   0 -> log-tf weight = 0.0000
tf =   1 -> log-tf weight = 1.0000
tf =   2 -> log-tf weight = 1.3010
tf =   5 -> log-tf weight = 1.6990
tf =  10 -> log-tf weight = 2.0000
tf = 100 -> log-tf weight = 3.0000


## Inverse document frequency

Vogliamo ora dare più peso ai termini rari e meno peso ai termini molto frequenti.

Per questo introduciamo la **inverse document frequency (idf)**:

$$
idf_t = \log_{10}\left(\frac{N}{df_t}\right)
$$

dove:
- \(N\) è il numero totale di documenti
- \(df_t\) è il numero di documenti in cui compare il termine \(t\)

### Intuizione
- se un termine compare quasi ovunque, è poco discriminativo
- se compare in pochi documenti, è più informativo

In [ ]:
def compute_idf(term, document_frequency, N):
    """
    Calcola l'inverse document frequency di un termine.

    Returns
    -------
    float
    """
    df = document_frequency.get(term, 0)

    if df == 0:
        return 0.0

    return math.log10(N / df)

In [ ]:
N = len(tokenized_documents)

print("Number of documents:", N)
print("\nSome idf values:\n")

for term in example_terms:
    print(f"{term:12s} -> idf = {compute_idf(term, document_frequency, N):.4f}")

Number of documents: 584

Some idf values:

graphic      -> idf = 0.4877
imag         -> idf = 0.6234
file         -> idf = 0.5678
format       -> idf = 0.9152
window       -> idf = 0.8972


## TF-IDF

Ora possiamo combinare i due ingredienti principali.

Il peso **tf-idf** di un termine \(t\) in un documento \(d\) è:

$$
w_{t,d} = (1 + \log_{10}(tf_{t,d})) \cdot \log_{10}\left(\frac{N}{df_t}\right)
$$

Questo peso:
- aumenta se il termine compare più volte nel documento
- aumenta se il termine è raro nella collezione

È una delle rappresentazioni più classiche dell’Information Retrieval.

In [ ]:
def compute_tf_idf_weights(tokens, document_frequency, N):
    """
    Calcola i pesi tf-idf di un documento.

    Returns
    -------
    dict[str, float]
        termine -> peso tf-idf
    """
    tf_counter = compute_term_frequencies(tokens)
    weights = {}

    for term, tf in tf_counter.items():
        weights[term] = log_tf_weight(tf) * compute_idf(term, document_frequency, N)

    return weights

In [ ]:
doc_id = 0
tf_idf_example = compute_tf_idf_weights(tokenized_documents[doc_id], document_frequency, N)

print(f"Some tf-idf weights for document {doc_id}:\n")
for term in ["cobra", "card", "inform", "driver", "graphic"]:
    print(f"{term:12s} -> tf-idf = {tf_idf_example.get(term, 0.0):.4f}")

Some tf-idf weights for document 0:

cobra        -> tf-idf = 4.0863
card         -> tf-idf = 1.5949
inform       -> tf-idf = 1.0823
driver       -> tf-idf = 1.2223
graphic      -> tf-idf = 0.0000


## Documenti come vettori sparsi

A questo punto, ogni documento può essere visto come un vettore molto grande:

- una dimensione per ogni termine del vocabolario
- peso tf-idf in quella dimensione
- zero nelle dimensioni dei termini assenti

Dal punto di vista concettuale, il documento è quindi un vettore in $\mathbb{R}^{|V|}$.

Dal punto di vista implementativo, però, conviene usare una **rappresentazione sparsa**:
- memorizziamo solo i termini con peso non nullo
- evitiamo di costruire subito grandi matrici dense inutili

In [ ]:
def build_document_vectors(tokenized_documents, document_frequency):
    """
    Costruisce la rappresentazione sparsa tf-idf di tutti i documenti.

    Returns
    -------
    dict[int, dict[str, float]]
        docID -> {termine: peso tf-idf}
    """
    N = len(tokenized_documents)
    document_vectors = {}

    for doc_id, tokens in tokenized_documents.items():
        document_vectors[doc_id] = compute_tf_idf_weights(tokens, document_frequency, N)

    return document_vectors

In [ ]:
document_vectors = build_document_vectors(tokenized_documents, document_frequency)

print("Number of document vectors:", len(document_vectors))
print("\nNumber of non-zero weighted terms in document 0:", len(document_vectors[0]))

sample_items = list(document_vectors[0].items())[:20]
print("\nFirst 20 weighted terms of document 0:\n")
for term, weight in sample_items:
    print(f"{term:15s} -> {weight:.4f}")

Number of document vectors: 584

Number of non-zero weighted terms in document 0: 65

First 20 weighted terms of document 0:

anyon           -> 0.7495
net             -> 1.5292
land            -> 1.9883
inform          -> 1.0823
cobra           -> 4.0863
two             -> 0.3448
zero            -> 0.3225
card            -> 1.5949
sticker         -> 2.7664
end             -> 1.3685
read            -> 0.8473
model           -> 1.1982
one             -> 0.2598
bio             -> 1.7250
havn            -> 2.7664
abl             -> 1.0504
find            -> 0.7621
anyth           -> 1.1132
get             -> 0.6297
hold            -> 1.8122


## Query come vettori

Nel Vector Space Model anche la query viene rappresentata come un vettore nello stesso spazio dei documenti.

Questo è un punto fondamentale:
- documenti e query vivono nello **stesso spazio vettoriale**
- possiamo quindi confrontarli direttamente

Anche per la query useremo una rappresentazione tf-idf semplice.

In [ ]:
def build_query_vector(query, document_frequency, N):
    """
    Costruisce il vettore tf-idf sparso di una query.

    Parameters
    ----------
    query : str

    Returns
    -------
    tuple
        (processed_tokens, query_weights)
    """
    processed_tokens = preprocess(query, is_query=True)
    query_weights = compute_tf_idf_weights(processed_tokens, document_frequency, N)

    return processed_tokens, query_weights

In [ ]:
query = "graphic file format"
processed_query, query_vector = build_query_vector(query, document_frequency, N)

print("Original query:", query)
print("Processed query tokens:", processed_query)

print("\nQuery weights:\n")
for term, weight in query_vector.items():
    print(f"{term:12s} -> {weight:.4f}")

Original query: graphic file format
Processed query tokens: ['graphic', 'file', 'format']

Query weights:

graphic      -> 0.4877
file         -> 0.5678
format       -> 0.9152


## Cosine similarity

Una volta rappresentati documenti e query come vettori, dobbiamo definire una misura di somiglianza.

Una scelta classica è la **cosine similarity**:

$$
\cos(q, d) = \frac{q \cdot d}{\|q\| \, \|d\|}
$$

### Perché la cosine?
Perché ci interessa confrontare la **direzione** dei vettori più che la loro lunghezza assoluta.

Questo è importante perché:
- documenti lunghi tendono naturalmente ad avere più termini
- non vogliamo favorire automaticamente i documenti più lunghi
- la normalizzazione aiuta a confrontare documenti di dimensioni diverse

In [ ]:
def dot_product_sparse(vec1, vec2):
    """
    Prodotto scalare tra due vettori sparsi rappresentati come dizionari.

    Returns
    -------
    float
    """
    # Iteriamo sul vettore più piccolo per essere più efficienti.
    if len(vec1) > len(vec2):
        vec1, vec2 = vec2, vec1

    score = 0.0
    for term, weight in vec1.items():
        if term in vec2:
            score += weight * vec2[term]

    return score


def l2_norm_sparse(vec):
    """
    Norma L2 di un vettore sparso.
    """
    return math.sqrt(sum(weight * weight for weight in vec.values()))


def cosine_similarity_sparse(query_vec, doc_vec):
    """
    Cosine similarity tra due vettori sparsi.

    Returns
    -------
    float
    """
    query_norm = l2_norm_sparse(query_vec)
    doc_norm = l2_norm_sparse(doc_vec)
    # MA VA FATTA CALCOLATA LA NORMA OGNI VOLTA???

    if query_norm == 0.0 or doc_norm == 0.0:
        return 0.0

    return dot_product_sparse(query_vec, doc_vec) / (query_norm * doc_norm)

In [ ]:
q = "graphic file format"
processed_query, query_vec = build_query_vector(q, document_frequency, N)

for doc_id in [0, 3, 15, 93]:
    score = cosine_similarity_sparse(query_vec, document_vectors[doc_id])
    print(f"doc {doc_id:3d} -> cosine score = {score:.4f}")

doc   0 -> cosine score = 0.0000
doc   3 -> cosine score = 0.0372
doc  15 -> cosine score = 0.1466
doc  93 -> cosine score = 0.1137


## Ranking dei documenti

Ora possiamo finalmente eseguire retrieval ranked:

1. preprocessiamo la query
2. costruiamo il suo vettore tf-idf
3. calcoliamo la cosine similarity con ogni documento
4. ordiniamo i documenti per score decrescente

Questo è il cuore del ranked retrieval nel Vector Space Model.

In [ ]:
def rank_documents(query, document_vectors, document_frequency, top_k=10):
    """
    Restituisce i top-k documenti per una query usando cosine similarity.

    Returns
    -------
    tuple
        (processed_query_tokens, ranked_results)

    dove ranked_results è una lista di tuple:
        (doc_id, score)
    """
    N = len(document_vectors)
    processed_query, query_vec = build_query_vector(query, document_frequency, N)

    results = []

    for doc_id, doc_vec in document_vectors.items():
        score = cosine_similarity_sparse(query_vec, doc_vec)
        if score > 0:
            results.append((doc_id, score))

    results.sort(key=lambda x: x[1], reverse=True)

    return processed_query, results[:top_k]

In [ ]:
def print_document(doc_id, documents, max_chars=1200):
    print(f"Document {doc_id}\n")
    print(documents[doc_id][:max_chars])

In [ ]:
query = "graphic file format"

processed_query, ranked_results = rank_documents(
    query,
    document_vectors,
    document_frequency,
    top_k=10
)

print("Original query:", query)
print("Processed query:", processed_query)

print("\nTop ranked documents:\n")
for rank, (doc_id, score) in enumerate(ranked_results, start=1):
    print(f"{rank:2d}. doc {doc_id:3d} -> score = {score:.4f}")

Original query: graphic file format
Processed query: ['graphic', 'file', 'format']

Top ranked documents:

 1. doc  35 -> score = 0.1943
 2. doc 202 -> score = 0.1748
 3. doc 210 -> score = 0.1548
 4. doc  15 -> score = 0.1466
 5. doc 469 -> score = 0.1460
 6. doc 431 -> score = 0.1444
 7. doc 414 -> score = 0.1443
 8. doc 299 -> score = 0.1376
 9. doc 116 -> score = 0.1339
10. doc 449 -> score = 0.1326


## Ispezione qualitativa del ranking

Come nel retrieval booleano, anche nel ranked retrieval è importante leggere alcuni documenti restituiti.

Questo passaggio permette di verificare:
- se i documenti più in alto nel ranking sembrano plausibili
- se i termini della query sono effettivamente centrali nel contenuto
- se il ranking sembra catturare la rilevanza meglio del semplice matching booleano

In [ ]:
if len(ranked_results) > 0:
    top_doc_id = ranked_results[0][0]
    print_document(top_doc_id, documents)

Document 35

From: agiacalo@nmsu.edu (Toni Giacalo)
Subject: need algorithm for reading and displaying bitmap files
Organization: New Mexico State University
Lines: 7
NNTP-Posting-Host: gauss.nmsu.edu
Keywords: GIF PCX BMP

I'm making a customized paint program in DOS and need an algorithm
for reading bitmap files like GIF, PCX, or BMP.  Does anyone have
such an algorithm?  I've tried copying one out of a book for reading
.PCX format but it doesn't work.  I will take an algorithm for any
format that can be created from Windows Paint.  
Thanks!
Toni



## Estensione: body e title con pesi diversi

Nei sistemi reali, non tutte le parti di un documento hanno la stessa importanza.

Per esempio:
- un termine presente nel **titolo** può essere molto informativo
- lo stesso termine nel **body** può essere meno discriminativo

Per questo, una possibile estensione consiste nel costruire rappresentazioni separate per:
- body
- title

e poi combinarle con pesi diversi.

In questo notebook mostriamo una versione semplice di questa idea.

In [ ]:
def extract_title_and_body(text):
    """
    Estrae in modo semplice titolo e body da un documento 20 Newsgroups.

    Se trova una riga Subject:, la usa come titolo.
    Il resto del testo, dopo la rimozione dell'header, viene usato come body.
    """
    title = ""

    for line in text.splitlines():
        if line.lower().startswith("subject:"):
            title = line[len("subject:"):].strip()
            break

    body = remove_header(text)
    return title, body

In [ ]:
titles = {}
bodies = {}

for doc_id, text in enumerate(documents):
    title, body = extract_title_and_body(text)
    titles[doc_id] = preprocess(title, is_query=True)
    bodies[doc_id] = preprocess(body, is_query=False)

print("Example title tokens:\n")
print(titles[0][:20])

print("\nExample body tokens:\n")
print(bodies[0][:40])

Example title tokens:

['cobra', 'two', 'zero', 'one', 'one', 'video', 'card', 'help']

Example body tokens:

['havn', 'abl', 'find', 'anyth', 'anyon', 'inform', 'get', 'hold', 'compani', 'produc', 'card', 'know', 'driver', 'pleas', 'let', 'know', 'far', 'tell', 'cga', 'card', 'take', 'two', 'one', 'six', 'bit', 'isa', 'slot', 'enabl', 'test', 'pattern', 'display', 'much', 'usuali', 'four', 'cga', 'color', 'least', 'one', 'six', 'count']


In [ ]:
def build_combined_document_vectors(title_tokens_by_doc, body_tokens_by_doc, alpha_title=2.0, alpha_body=1.0):
    """
    Costruisce vettori documento combinando title e body con pesi diversi.

    Parameters
    ----------
    alpha_title : float
        Peso moltiplicativo per i termini del titolo.
    alpha_body : float
        Peso moltiplicativo per i termini del body.

    Returns
    -------
    tuple
        (combined_vectors, combined_df)
    """
    combined_docs = {}

    for doc_id in body_tokens_by_doc:
        combined_docs[doc_id] = {
            "title": title_tokens_by_doc[doc_id],
            "body": body_tokens_by_doc[doc_id]
        }

    # Per il df consideriamo la presenza del termine nel documento,
    # indipendentemente dalla sezione in cui compare.
    combined_tokenized_documents = {}
    for doc_id in combined_docs:
        combined_tokenized_documents[doc_id] = combined_docs[doc_id]["title"] + combined_docs[doc_id]["body"]

    combined_df = build_document_frequency(combined_tokenized_documents)
    N = len(combined_tokenized_documents)

    combined_vectors = {}

    for doc_id in combined_docs:
        title_weights = compute_tf_idf_weights(combined_docs[doc_id]["title"], combined_df, N)
        body_weights = compute_tf_idf_weights(combined_docs[doc_id]["body"], combined_df, N)

        merged = defaultdict(float)

        for term, weight in title_weights.items():
            merged[term] += alpha_title * weight

        for term, weight in body_weights.items():
            merged[term] += alpha_body * weight

        combined_vectors[doc_id] = dict(merged)

    return combined_vectors, combined_df

In [ ]:
combined_document_vectors, combined_df = build_combined_document_vectors(
    titles,
    bodies,
    alpha_title=2.0,
    alpha_body=1.0
)

query = "graphic file format"

processed_query, ranked_combined = rank_documents(
    query,
    combined_document_vectors,
    combined_df,
    top_k=10
)

print("Top ranked documents with title/body weighting:\n")
for rank, (doc_id, score) in enumerate(ranked_combined, start=1):
    print(f"{rank:2d}. doc {doc_id:3d} -> score = {score:.4f}")

Top ranked documents with title/body weighting:

 1. doc 210 -> score = 0.8375
 2. doc 202 -> score = 0.3473
 3. doc 414 -> score = 0.3342
 4. doc 116 -> score = 0.2634
 5. doc 431 -> score = 0.2468
 6. doc 323 -> score = 0.2338
 7. doc 377 -> score = 0.2225
 8. doc  35 -> score = 0.2188
 9. doc 337 -> score = 0.2004
10. doc 449 -> score = 0.1969


Se elaboriamo i documenti senza "dividere" titolo e corpo del documento

In [ ]:
print_document(ranked_results[0][0], documents)

Document 35

From: agiacalo@nmsu.edu (Toni Giacalo)
Subject: need algorithm for reading and displaying bitmap files
Organization: New Mexico State University
Lines: 7
NNTP-Posting-Host: gauss.nmsu.edu
Keywords: GIF PCX BMP

I'm making a customized paint program in DOS and need an algorithm
for reading bitmap files like GIF, PCX, or BMP.  Does anyone have
such an algorithm?  I've tried copying one out of a book for reading
.PCX format but it doesn't work.  I will take an algorithm for any
format that can be created from Windows Paint.  
Thanks!
Toni



Se diamo maggiore peso al titolo

In [ ]:
print_document(ranked_combined[0][0], documents)

Document 210

From: rubery@saturn.aitc.rest.tasc.com. (Dan Rubery)
Subject: Graphic Formats
Organization: TASC
Lines: 7
NNTP-Posting-Host: saturn.aitc.rest.tasc.com

I am writing some utilies to convert Regis and Tektonic esacpe sequences  
into some useful formats. I would rather not have to goto a bitmap format.  
I can convert them to Window Meta FIles easily enough, but I would rather  
convert them to Corel Draw, .CDR, or MS Power Point, .PPT, files.  
Microsoft would not give me the format. I was wondering if anybody out  
there knows the formats for these two applications.




## Osservazione finale

Questa estensione mostra un punto importante:

nel ranked retrieval non conta solo:
- quali termini compaiono
- quante volte compaiono

Conta anche **dove** compaiono.

In un sistema reale si possono assegnare pesi diversi a:
- titolo
- abstract
- body
- anchor text
- campi strutturati

Questo rende il ranking più flessibile e più vicino alle esigenze applicative reali.

## Riassunto del laboratorio

In questo laboratorio abbiamo introdotto il **ranked retrieval** nel quadro del **Vector Space Model**.

### Abbiamo visto come
- rappresentare i documenti con il modello **bag of words**
- usare la **term frequency**
- usare la **document frequency**
- costruire i pesi **tf-idf**
- rappresentare anche la query come vettore
- confrontare query e documenti con la **cosine similarity**
- ottenere un ranking dei documenti

### Abbiamo anche osservato che
il ranked retrieval permette di superare un limite importante del retrieval booleano:

- non ci fermiamo più a decidere se un documento matcha o no
- cerchiamo invece di stimare **quanto** il documento sia rilevante per la query

### Messaggio chiave
Il Vector Space Model trasforma il problema del retrieval in un problema geometrico:

- documenti e query diventano vettori
- i pesi dei termini modellano l'importanza lessicale
- la similarità tra vettori diventa una stima di rilevanza

## Esercizi finali

### Esercizio 1 — Confrontare boolean retrieval e ranked retrieval
Scegli alcune query e confronta:
- i documenti restituiti dal sistema booleano
- i top documenti restituiti dal sistema ranked

**Domande guida**
- il ranking sembra più utile per l’utente?
- i primi documenti sembrano plausibili?
- ci sono query per cui il booleano è troppo rigido?

### Esercizio 2 — Analizzare l’effetto del peso del titolo
Modifica i pesi assegnati a titolo e body, per esempio:
- `alpha_title = 1.0`, `alpha_body = 1.0`
- `alpha_title = 2.0`, `alpha_body = 1.0`
- `alpha_title = 3.0`, `alpha_body = 1.0`

**Domande guida**
- come cambia il ranking?
- aumentare il peso del titolo sembra utile?
- in quali casi il titolo è davvero informativo?